# Issue8回测系统补充
先运行固定的研究对照，再读同池差额、未完成目标和残仓。此研究不证明因子盈利。

In [ ]:
from pathlib import Path
import os, sys, subprocess
ROOT=Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0,str(ROOT))
subprocess.run([sys.executable,"run_issue8_review.py"],check=True)

In [ ]:
import pandas as pd
display(pd.read_csv(ROOT/"reports/issue8_upgrade/matched_comparisons.csv"))
display(pd.read_csv(ROOT/"reports/issue8_upgrade/5dr_slip0/residual.csv"))

## 用于自己的策略
先生成面板Target，设置预知的退出交易日历，再通过run_traced执行；退出期禁止加仓。

In [ ]:
import qbt
d=qbt.load()
m=qbt.build_market(d,log=None)
ctx=qbt.Context(d,m)
u=ctx.universe(min_amt=5e7,min_listed=60,not_st=True)
target=ctx.long(u.bottom(ctx.ret(5),100),gross=.98)
target,plan=qbt.exit_schedule(target,d.dates,"2026-09-30",sessions=20)
result,attempts=qbt.run_traced(d,target,start="2025-01-02",end="2026-09-30",capital=1e9,market=m,stop_buys_from=plan["first_execution"])
display(qbt.report(result))
display(pd.DataFrame(qbt.residual_inventory(result)))